# 3차 실험 모델 선정
2차 실험 run의 클래스별 AP를 그 클래스의 train·val·test 객체 수와 한 그림에 놓고, train 객체가 적은 클래스(희소 절반)와 많은 클래스(빈번 절반)의 AP 차이를 모델끼리 비교합니다. 3차 실험(오버/언더샘플링, 클래스별 손실 가중치)에 쓸 모델을 고르는 근거로 씁니다.

**준비**
- 내 Drive에 `.env`를 두고, 그 안에 `DRIVE_BASE=<pill data 폴더 경로>`를 적습니다. 예) `DRIVE_BASE=/content/drive/MyDrive/pill data`
- `.env`에는 API 키도 들어가므로 팀 공유 폴더(`pill data`)가 아니라 내 Drive에 둡니다. 키 이름은 [.env.example](../../.env.example)에서 확인합니다.

**쓰는 법**
1. **각자 바꿀 값** 셀에서 `ENV_FILE`을 내 `.env` 경로로 맞추고, `RUNS`에 비교할 2차 run 폴더(`DRIVE_BASE` 기준)와 그 run의 방법(v2, v3, v4)을 적습니다. 폴더에는 채점 때 저장한 `scores.json`이 있어야 합니다.
2. 위에서부터 실행합니다. 학습을 하지 않으므로 GPU 런타임이 필요 없습니다.

**그림 읽는 법**
- 방법마다 한 장을 그립니다. 한 줄이 한 클래스이고, 그 방법의 train 객체가 적은 클래스부터 위에서 아래로 놓습니다.
- 왼쪽 막대는 클래스별 train·test·val 객체 수, 오른쪽 점은 run마다 그 클래스의 AP입니다.
- 가로선 위가 희소 절반, 아래가 빈번 절반입니다(74개 클래스면 37개씩). 세로선은 각 절반의 평균 AP이고, 두 세로선의 차이가 gap입니다.
- Colab에는 한글 글꼴이 없어서 그림 안의 글자는 영어로 씁니다.

**판단 기준**
- 같은 방법끼리 비교합니다. 방법마다 train 객체 수가 달라서, 방법이 다른 run의 gap은 비교하지 않습니다.
- ① gap(빈번 절반 AP − 희소 절반 AP)이 큰 모델: 불균형의 영향을 크게 받아 3차에서 개선할 여지가 큽니다.
- ② val 여유(1 − val)가 있는 모델: val이 1에 가까우면 3차의 개선이 val에 드러나지 않습니다.
- RT-DETR는 클래스별 손실 가중치(`cls_pw`)가 손실에 반영되지 않아, 고르면 샘플링 조건만 돌립니다. (`src/pill_detection/balance.py` 모듈 docstring)

In [ ]:
# 각자 바꿀 값. 값을 바꾸면 이 셀부터 다시 실행한다
ENV_FILE = '/content/drive/MyDrive/.secret/pill_object_detection.env'   # 내 Drive의 .env. DRIVE_BASE(pill data 폴더 경로)가 들어 있어야 한다
BRANCH = 'feat/41-exp3-balance'            # balance.py가 있는 브랜치. main에 병합되면 'main'으로 바꾼다
RUNS = {   # run 이름: (DRIVE_BASE 아래 run 폴더, 그 run을 학습한 방법). scores.json이 있는 폴더를 직접 적는다
    'v4_frcnn_e20_s0': ('cj_runs/v4_frcnn_e20_s0_b16_fast', 'v4'),
    'v4_frcnn_e10_s0': ('cj_runs/v4_frcnn_e10_s0_b16_fast', 'v4'),
    'v4_frcnn_e5_s0': ('cj_runs/v4_frcnn_e5_s0_b16_fast', 'v4'),
    'v3_frcnn_e20_s0': ('rw_runs/v3_frcnn_e20_s0', 'v3'),
    'v3_frcnn_e10_s0': ('rw_runs/v3_frcnn_e10_s0', 'v3'),
    'v3_frcnn_e5_s0': ('rw_runs/v3_frcnn_e5_s0', 'v3'),
    'v2_frcnn_e20_s0': ('rw_runs/v2_frcnn_e20_s0', 'v2'),
    'v2_frcnn_e10_s0': ('cj_runs/v2_frcnn_e10_s0_b16_fast', 'v2'),
    'v2_frcnn_e5_s0': ('cj_runs/v2_frcnn_e5_s0_b16_fast', 'v2'),
    # 'v1_frcnn_e20_s0': ('cj_runs/v1_frcnn_e20_s0_b16_fast', 'v1'), # 학습 끝나면 주석 풀기
    'v1_frcnn_e10_s0': ('cj_runs/v1_frcnn_e10_s0_b16_fast', 'v1'),
    'v1_frcnn_e5_s0': ('cj_runs/v1_frcnn_e5_s0_b16_fast', 'v1'),
    # 'v3_yolo26s': ('<이름>_runs/v3_yolo26s', 'v3'),
    # 'v3_rtdetr-l': ('<이름>_runs/v3_rtdetr-l', 'v3'),
}

In [ ]:
# 세션 시작 (Drive 연결, .env에서 DRIVE_BASE 읽기, 레포 받기). 학습을 하지 않으므로 W&B 로그인은 하지 않는다
import os
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

!pip install -q python-dotenv==1.2.4
from dotenv import load_dotenv

if not Path(ENV_FILE).is_file():
    raise FileNotFoundError(f'.env가 없습니다: {ENV_FILE}\n'
                            '[해결] 내 Drive에 .env를 올리고(.env.example 참고), 각자 바꿀 값 셀의 ENV_FILE을 그 경로로 고친 뒤 그 셀부터 다시 실행하세요.')
load_dotenv(ENV_FILE, override=True)
DRIVE_BASE = Path(os.environ.get('DRIVE_BASE', '').strip())
if not os.environ.get('DRIVE_BASE', '').strip() or not DRIVE_BASE.is_dir():
    raise FileNotFoundError(f'DRIVE_BASE가 비어 있거나 Drive에 없는 폴더입니다: "{os.environ.get("DRIVE_BASE", "")}"\n'
                            '[해결] .env에 DRIVE_BASE=<pill data 폴더 경로>를 적으세요. 예) DRIVE_BASE=/content/drive/MyDrive/pill data')

REPO = '/content/pill-object-detection'
!test -d {REPO} || git clone -q https://github.com/codeit-ai-15th/pill-object-detection.git {REPO}
%cd {REPO}
!git fetch -q origin {BRANCH} && git switch -q -C {BRANCH} FETCH_HEAD && git log -1 --format='%h %s'

import sys; sys.path.insert(0, f'{REPO}/src')

In [ ]:
# 모델 선정 그림: 방법마다 한 장. 클래스를 train 객체가 적은 순서로 위에서부터 놓고, 같은 줄에 객체 수(왼쪽)와 run별 AP(오른쪽)를 그린다
import json

import matplotlib.pyplot as plt
import numpy as np

from pill_detection.balance import class_ap_rows, imbalance_summary

CLASSES = f'{REPO}/data/datasets/classes.csv'
missing = [run_dir for run_dir, _ in RUNS.values() if not (DRIVE_BASE / run_dir / 'scores.json').is_file()]
if missing:
    raise FileNotFoundError(f'scores.json이 없는 run 폴더가 있습니다(DRIVE_BASE 기준): {missing}\n'
                            '[해결] RUNS의 경로를 Drive에서 확인하고, 채점(val 채점 셀)을 마친 run인지 확인하세요.')

by_variant = {}   # 방법: [(run 이름, scores, 요약)]. 방법마다 train 객체 수가 달라 같은 방법끼리만 한 그림에 놓는다
for name, (run_dir, variant) in RUNS.items():
    scores = json.loads((DRIVE_BASE / run_dir / 'scores.json').read_text(encoding='utf-8'))
    s = imbalance_summary(scores['per_class'], CLASSES, variant)
    print(f"{name} ({variant}) | val {scores['val']:.4f} (여유 {1 - scores['val']:.4f}) | "
          f"희소 절반 {s['rare_ap']:.4f}  빈번 절반 {s['frequent_ap']:.4f}  gap {s['gap']:.4f}  [{s['n_classes']}개 클래스]")
    by_variant.setdefault(variant, []).append((name, scores, s))

SURFACE, INK, MUTED, GRID, AXIS = '#fcfcfb', '#52514e', '#898781', '#e1e0d9', '#c3c2b7'
COUNTS = [('train', 'n_train', '#2a78d6'), ('test', 'n_test', '#eb6834'), ('val', 'n_val', '#1baf7a')]
for variant, runs in by_variant.items():
    rows = class_ap_rows(runs[0][1]['per_class'], CLASSES, variant)
    y, half = np.arange(len(rows)), len(rows) // 2   # imbalance_summary와 같게 위 half개가 희소 절반
    lowest = min(min(scores['per_class'].values()) for _, scores, _ in runs)
    fig, axes = plt.subplots(1, len(runs) + 1, sharey=True, figsize=(4.5 + 2.4 * len(runs), 0.2 * len(rows) + 1.5),
                             gridspec_kw={'width_ratios': [2] + [1] * len(runs)}, facecolor=SURFACE)
    for k, (label, key, color) in enumerate(COUNTS):
        axes[0].barh(y + (k - 1) * 0.28, [r[key] for r in rows], height=0.24, color=color, label=label)
    axes[0].set_yticks(y, [r['k_code'] for r in rows], fontsize=8)
    axes[0].set_title(f'objects per class ({variant})', loc='left', fontsize=9)
    axes[0].set_xlabel('objects')
    axes[0].legend(loc='upper right', frameon=False, fontsize=9)
    for text, row, va in (('rare half', half - 1, 'bottom'), ('frequent half', half, 'top')):
        axes[0].text(0.98, row, text, transform=axes[0].get_yaxis_transform(), ha='right', va=va, fontsize=9, color=INK)
    for ax, (name, scores, s) in zip(axes[1:], runs):
        ap = {int(k): v for k, v in scores['per_class'].items()}
        ax.scatter([ap[r['category_id']] for r in rows], y, s=30, color=INK, edgecolors=SURFACE, linewidths=1.5, zorder=3)
        for mean, start, end, row, va in ((s['rare_ap'], -0.5, half - 0.5, half - 1, 'bottom'),
                                          (s['frequent_ap'], half - 0.5, len(rows) - 0.5, half, 'top')):
            ax.vlines(mean, start, end, color='#0b0b0b', lw=1)   # 그 절반의 평균 AP. 두 세로선의 차이가 gap이다
            ax.text(mean, row, f' mean {mean:.3f}', ha='left', va=va, fontsize=8, color=INK)
        ax.set_xlim(max(0.0, lowest - 0.02), 1.01)
        ax.set_title(f'{name}\ngap {s["gap"]:.4f}', loc='left', fontsize=9)
        ax.set_xlabel('AP')
    for ax in axes:
        ax.axhline(half - 0.5, color=MUTED, lw=1)
        ax.grid(axis='x', color=GRID, lw=0.8)
        ax.set_axisbelow(True)
        ax.set_facecolor(SURFACE)
        ax.spines[['top', 'right']].set_visible(False)
        ax.spines[['left', 'bottom']].set_color(AXIS)
        ax.tick_params(colors=MUTED, labelcolor=INK)
    axes[0].set_ylim(len(rows) - 0.5, -0.5)   # 위아래를 뒤집어 train 객체가 가장 적은 클래스를 맨 위에 둔다. sharey라 모든 칸에 적용된다
    fig.tight_layout()
    plt.show()